Author: Catherine Gatt

Date: 24/02/2026

Content: 
computes de genes between workers adn otehr castes and saves results. 

In [8]:
# --- imports ---
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc

# -----------------------------
# CONFIG
# -----------------------------
cell_type_col = "paper_cell_type_annotation"
caste_col = "caste"

reference_caste = "worker"
target_castes = ["queen", "king", "soldier"]   # comparisons performed: each vs worker

method = "wilcoxon"
alpha = 0.05
min_cells_per_group = 2

OUTDIR = Path("derived")
OUTDIR.mkdir(exist_ok=True, parents=True)
DE_PATH = OUTDIR / "differential_expression.parquet"

print("Will write:", DE_PATH.resolve())

h5ad_file = "/home/catherine/phd/projects/termites/data/znev/combined_no_norm_clustered.h5ad"
adata = sc.read_h5ad(h5ad_file)



Will write: /home/catherine/phd/projects/termites/code/znev_analysis/paper_code/fig_2/derived/differential_expression.parquet


In [9]:
def compute_differential_expression(
    adata,
    *,
    cell_type_col: str,
    caste_col: str,
    reference_caste: str,
    target_castes: list[str],
    method: str = "wilcoxon",
    min_cells_per_group: int = 2,
) -> pd.DataFrame:
    """Return long-format DE results for (cell_type, caste vs reference) comparisons."""
    rows = []
    cell_types = list(pd.unique(adata.obs[cell_type_col]))
    for cell_type in cell_types:
        adata_ct = adata[adata.obs[cell_type_col] == cell_type].copy()
        # must have reference cells for this cell type
        n_ref_total = int((adata_ct.obs[caste_col] == reference_caste).sum())
        for caste in target_castes:
            subset = adata_ct[adata_ct.obs[caste_col].isin([caste, reference_caste])].copy()
            n_caste = int((subset.obs[caste_col] == caste).sum())
            n_ref = int((subset.obs[caste_col] == reference_caste).sum())

            if n_caste < min_cells_per_group or n_ref < min_cells_per_group:
                continue

            subset.obs["caste_group"] = (subset.obs[caste_col] == caste).map({True: "caste", False: "worker"}).astype("category")

            sc.tl.rank_genes_groups(
                subset,
                groupby="caste_group",
                groups=["caste"],
                reference="worker",
                method=method,
                pts=True,
            )

            de = sc.get.rank_genes_groups_df(subset, group="caste").copy()

            # Standardize p-value columns (Scanpy can give pvals and pvals_adj)
            if "pvals" not in de.columns:
                de["pvals"] = np.nan
            if "pvals_adj" not in de.columns:
                de["pvals_adj"] = np.nan
            if "scores" not in de.columns:
                de["scores"] = np.nan
            if "logfoldchanges" not in de.columns:
                de["logfoldchanges"] = np.nan

            # Add metadata columns
            de["cell_type"] = cell_type
            de["caste"] = caste
            de["reference_caste"] = reference_caste
            de["n_caste_cells"] = n_caste
            de["n_reference_cells"] = n_ref
            de["method"] = method

            # If Scanpy provides fraction expressing columns, keep them (names differ by version)
            # Common columns: 'pct_nz_group', 'pct_nz_reference', or 'pts', 'pts_rest'
            rows.append(de)

    if not rows:
        return pd.DataFrame()

    out = pd.concat(rows, ignore_index=True)

    # Make sure gene IDs are strings
    if "names" in out.columns:
        out["names"] = out["names"].astype(str)

    # Convenience: significance flags (use adj if available, else raw)
    pcol = "pvals_adj" if out["pvals_adj"].notna().any() else "pvals"
    out["p_col_used"] = pcol
    out["is_sig"] = out[pcol] < alpha
    out["direction"] = np.where(out["logfoldchanges"] > 0, "up", "down")

    # Helpful ordering
    keep_first = ["cell_type","caste","reference_caste","n_caste_cells","n_reference_cells","names",
                  "scores","logfoldchanges","pvals","pvals_adj","p_col_used","is_sig","direction","method"]
    cols = keep_first + [c for c in out.columns if c not in keep_first]
    return out[cols]


In [10]:
# -----------------------------
# Run + save
# -----------------------------
differential_expression = compute_differential_expression(
    adata,
    cell_type_col=cell_type_col,
    caste_col=caste_col,
    reference_caste=reference_caste,
    target_castes=target_castes,
    method=method,
    min_cells_per_group=min_cells_per_group,
)

print(differential_expression.shape)
display(differential_expression.head())

differential_expression.to_parquet(DE_PATH, index=False)
print("Saved:", DE_PATH.resolve())


(941952, 15)


,cell_type,caste,reference_caste,n_caste_cells,n_reference_cells,names,scores,logfoldchanges,pvals,pvals_adj,p_col_used,is_sig,direction,method,pct_nz_group
0,T5,queen,worker,219,289,Znev00014135,13.997790,4.173733,1.607918e-44,2.294820e-40,pvals_adj,True,up,wilcoxon,0.894977
1,T5,queen,worker,219,289,Znev00014136,8.678813,1.539527,3.999193e-18,1.426912e-14,pvals_adj,True,up,wilcoxon,0.926941
2,T5,queen,worker,219,289,Znev00014164,7.504244,1.897631,6.178412e-14,1.469638e-10,pvals_adj,True,up,wilcoxon,0.849315
3,T5,queen,worker,219,289,Znev00011303,7.049247,2.660425,1.798884e-12,3.209209e-09,pvals_adj,True,up,wilcoxon,0.538813
4,T5,queen,worker,219,289,Znev00013159,6.906736,2.757390,4.959302e-12,7.864351e-09,pvals_adj,True,up,wilcoxon,0.497717


Saved: /home/catherine/phd/projects/termites/code/znev_analysis/paper_code/fig_2/derived/differential_expression.parquet
